# Building main2.py From Scratch
**A step-by-step teaching guide**

---

This notebook walks you through every section of the finance app in the order you would write it yourself. Each section explains the *why* before the *how*.

### What we are building
A Streamlit web app that:
- Accepts one or more bank CSV exports (one per month)
- Automatically detects which bank the CSV came from
- Normalises any bank's column names into a standard format
- Lets you categorise spending and saves those choices for future uploads
- Shows expense summaries and charts per month
- Compares spending across months

### Prerequisites
```
pip install streamlit pandas plotly
```

---
## Section 1 — How Streamlit Works (Read This First)

Before writing a single line, you need to understand Streamlit's core behaviour. It is different from normal Python scripts.

### The rerun model
Every time the user interacts with anything — clicks a button, changes a dropdown, uploads a file — Streamlit **re-runs the entire script from top to bottom**. Not just the part that changed. Everything.

This means:
- Variables defined in the script are reset on every rerun
- You cannot store data in a normal variable and expect it to survive a button click
- You must use **`st.session_state`** to persist data across reruns

### `st.session_state`
`st.session_state` is a dictionary-like object that survives reruns. Anything you store in it stays there until the browser tab is closed or you explicitly delete it.

```python
# Wrong — lost on every rerun
my_data = []

# Right — persists across reruns
if "my_data" not in st.session_state:
    st.session_state.my_data = []
```

### Widget keys
Every interactive widget (button, text input, data editor, date picker) must have a **unique `key`** string. If the same function is called multiple times in one rerun (e.g. rendering multiple months), every widget inside it would have a duplicate key and crash unless you suffix the key with something unique.

```python
# Crashes if called twice — duplicate key
st.button("Apply", key="apply_btn")

# Works — unique per month
st.button("Apply", key=f"apply_btn_{month_key}")
```

### Execution order matters
The script runs top to bottom. If you compute a variable on line 10 and a button handler on line 50 modifies the data that variable came from, the variable on line 10 still holds the **old value** for that rerun. The new value only appears on the **next** rerun. This is why the expense summary must be computed **after** the Apply Changes button handler, not before.

---
## Section 2 — Imports and Page Config

In [ ]:
import streamlit as st   # UI framework — every visible element comes from here
import pandas as pd      # reads and manipulates the CSV data
import numpy as np       # numerical operations (used indirectly by pandas/plotly)
import plotly.express as px  # charts — pie, bar, line
import json              # read/write .json files for persistence
import os                # check if a file exists on disk
import base64            # encode the background image as a string for CSS

### Page config
`st.set_page_config` **must be the first Streamlit call** in the script. It sets the browser tab title, icon, and layout. `layout='wide'` makes the app use the full screen width instead of a narrow centred column.

In [ ]:
st.set_page_config(page_title="Simple Finance Tracking App", page_icon="🤑", layout='wide')

---
## Section 3 — Background Image and Custom CSS

Streamlit doesn't have a built-in background image option. We inject raw CSS using `st.markdown` with `unsafe_allow_html=True`.

A local image file can't be referenced by file path in CSS (the browser doesn't have access to your filesystem). The solution is to **encode the image as a base64 string** and embed it directly in the CSS `url()`. This converts the image bytes into a long text string that lives entirely inside the HTML page.

### Step 1 — Encode the image

In [ ]:
def get_base64_image(path):
    with open(path, "rb") as f:          # open in binary read mode
        return base64.b64encode(f.read()).decode()  # bytes → base64 text

img = get_base64_image("image finance tracker.webp")
# img is now a very long string like "UklGRv4AAABXRUJQVlA4..."

### Step 2 — Inject CSS

Key CSS concepts used here:

| CSS property | What it does |
|---|---|
| `background-size: cover` | Scales the image to fill the entire element, cropping edges if needed |
| `background-attachment: fixed` | Image stays still while the page scrolls |
| `rgba(r, g, b, alpha)` | Colour with transparency. Alpha 0 = invisible, 1 = solid |
| `backdrop-filter: blur(10px)` | Frosted glass effect — blurs whatever is behind the element |
| `!important` | Overrides Streamlit's own inline styles |

Note the double braces `{{` in the f-string — a single `{` would be interpreted as an f-string variable. `{{` is the way to write a literal `{` in an f-string.

In [ ]:
st.markdown(f"""
<style>
.stApp {{
    background-image: url("data:image/webp;base64,{img}");
    background-size: cover;
    background-position: center;
    background-attachment: fixed;
}}
[data-testid="stAppViewContainer"] {{
    background: rgba(0, 0, 0, 0.30);   /* dark overlay on the whole app */
}}
.block-container {{
    background: rgba(14, 17, 23, 0.55); /* frosted card on the main content */
    border-radius: 16px;
    border: 1px solid rgba(255, 255, 255, 0.08);
    backdrop-filter: blur(10px);
    -webkit-backdrop-filter: blur(10px);
    padding: 2rem 3rem !important;
}}
[data-testid="stHeader"] {{
    background: rgba(0, 0, 0, 0.0);    /* transparent top bar */
}}
</style>
""", unsafe_allow_html=True)

---
## Section 4 — Bank Detection System

Different banks export CSV files with different column names. Before we can read the data, we need to know which bank it came from so we know which columns to use.

### 4a — Known bank formats dictionary

For each known bank we store a `fmt` dict with:
- `date_col` — the column name that holds the transaction date
- `date_format` — the strftime format string to parse that date (e.g. `"%d/%m/%Y"`)
- `description_col` — the column name that holds the transaction description
- `amount_col` — the column name that holds the amount (used for most banks)
- `debit_col` / `credit_col` — used instead of `amount_col` for banks like HSBC that split payments and receipts into two columns

In [ ]:
BANK_FORMATS = {
    "Revolut": {
        "date_col":        "Started Date",
        "date_format":     "%d/%m/%Y %H:%M",
        "description_col": "Description",
        "amount_col":      "Amount",
    },
    "Monzo": {
        "date_col":        "Date",
        "date_format":     "%d/%m/%Y",
        "description_col": "Name",
        "amount_col":      "Amount",
    },
    "HSBC": {
        "date_col":        "Date",
        "date_format":     "%d/%m/%Y",
        "description_col": "Payee",
        "debit_col":       "Paid out",   # payments
        "credit_col":      "Paid in",    # receipts
        # No amount_col — handled separately in normalise_df
    },
}

### 4b — Fingerprint

A fingerprint is a single string that uniquely represents a CSV's column set. It is made by sorting the column names alphabetically and joining them with commas.

Why sort? Because column order can vary between exports of the same bank. Sorting makes the fingerprint consistent regardless of column order.

```
columns = ["Amount", "Date", "Name", "Emoji"]
fingerprint → "Amount,Date,Emoji,Name"
```

The fingerprint is used as a **dictionary key** in `bank_mappings.json` so we can look up a previously saved column mapping for an unknown bank.

In [ ]:
def make_fingerprint(columns):
    return ",".join(sorted(c.strip() for c in columns))

# Example:
make_fingerprint(["Amount", " Date", "Name", "Emoji"])  # → "Amount,Date,Emoji,Name"

### 4c — `detect_bank`

Detects the bank by looking for **signature columns** — column names that are unique to one bank's export format. We don't need to match every column, just enough to be certain.

For example, Revolut is the only bank that exports a column called `"Started Date"` alongside `"Completed Date"`. If both exist, it's Revolut.

In [ ]:
def detect_bank(columns):
    col_set = set(columns)  # set lookup is O(1), faster than list lookup
    if "Started Date" in col_set and "Completed Date" in col_set:
        return "Revolut"
    if "Emoji" in col_set and "Name" in col_set:
        return "Monzo"
    if "Memo" in col_set and "Subcategory" in col_set:
        return "Barclays"
    if "Paid out" in col_set and "Paid in" in col_set:
        return "HSBC"
    if "Counter Party" in col_set and "Reference" in col_set:
        return "Starling"
    return None  # unknown bank

### 4d — Saving unknown bank mappings

When `detect_bank` returns `None`, we ask the user to manually map the columns once. We then save that mapping to `bank_mappings.json` keyed by fingerprint, so **next time the same bank's CSV is uploaded, it is recognised automatically**.

Structure of `bank_mappings.json`:
```json
{
  "Amount,Date,Reference,Type,Value": {
    "date_col": "Date",
    "date_format": "%d/%m/%Y",
    "description_col": "Reference",
    "amount_col": "Value"
  }
}
```

In [ ]:
def load_bank_mappings():
    if os.path.exists("bank_mappings.json"):
        with open("bank_mappings.json", "r") as f:
            return json.load(f)
    return {}  # first run — no saved mappings yet

def save_bank_mappings(mappings):
    with open("bank_mappings.json", "w") as f:
        json.dump(mappings, f, indent=2)

---
## Section 5 — Categories System

Categories let the user label each outflow transaction (e.g. "Food", "Transport"). The system has two parts:
1. **Keyword matching** — if a transaction description matches a saved keyword, it is auto-categorised
2. **Persistence** — categories and their keywords are saved to `categories.json` so they carry over to future uploads

### 5a — Initialising categories in session state

The `if "categories" not in st.session_state` guard prevents resetting the dict on every rerun. But we ALSO re-load from JSON every rerun, because the JSON file is always the source of truth.

In [ ]:
# Initialise with defaults only if session state is empty
if "categories" not in st.session_state:
    st.session_state.categories = {"Uncategorized": [], "New category": []}

# Overwrite with saved data every rerun — JSON is the source of truth
if os.path.exists("categories.json"):
    with open("categories.json", "r") as f:
        st.session_state.categories = json.load(f)

# Structure:
# {
#   "Uncategorized": [],
#   "Food":          ["tesco", "greggs", "mcdonalds"],
#   "Transport":     ["tfl", "uber", "national rail"]
# }

### 5b — `categorize_transactions`

Called immediately after loading a CSV. Adds a `Category` column to every transaction.

The logic:
1. Start every row as `"Uncategorized"`
2. For each category that has keywords, loop through every outflow transaction
3. If the transaction's description **exactly matches** (case-insensitive) a saved keyword, assign that category

Important: we skip inflows (`Amount > 0`) because inflows are categorised separately with a per-transaction override system.

In [ ]:
def categorize_transactions(df):
    df["Category"] = "Uncategorized"  # default for every row

    for category, keywords in st.session_state.categories.items():
        if category == "Uncategorized" or not keywords:
            continue  # nothing to match against

        lowered_keywords = [kw.lower().strip() for kw in keywords]

        for idx, row in df.iterrows():
            if row["Amount"] > 0:
                continue  # skip inflows
            if row["Description"].lower().strip() in lowered_keywords:
                df.at[idx, "Category"] = category
                # .at[idx, col] is the fastest way to set a single cell by index

    return df

### 5c — `add_keyword_to_category`

Called when the user clicks Apply Changes on the outflow table. If they changed a row's category, the transaction's description is saved as a keyword for that category.

This is the **learning mechanism**: next time a CSV is uploaded, any transaction with the same description is automatically assigned the same category.

In [ ]:
def save_categories():
    with open("categories.json", "w") as f:
        json.dump(st.session_state.categories, f)

def add_keyword_to_category(category, keyword):
    keyword = keyword.strip()
    if keyword and keyword not in st.session_state.categories[category]:
        st.session_state.categories[category].append(keyword)
        save_categories()  # write to disk immediately
        return True
    return False  # already exists — no duplicate

---
## Section 6 — Inflow Overrides

Outflows use keyword matching: the same description always maps to the same category (e.g. "Tesco" is always "Food").

Inflows can't use that approach. The same description (e.g. "Transfer from Ali") could be salary one month and a refund another month. Each inflow transaction needs its own individual category memory.

### The key
Each inflow transaction is identified by a **unique key** combining its date, description, and amount:
```
"2026-05-01|Transfer from Ali|500.0"
```
Two transfers from Ali on different dates get different keys, so categorising one doesn't affect the other.

In [ ]:
# Load saved overrides at startup
if os.path.exists("inflow_overrides.json"):
    with open("inflow_overrides.json", "r") as f:
        inflow_overrides = json.load(f)
else:
    inflow_overrides = {}

# Structure of inflow_overrides.json:
# {
#   "2026-05-01|Transfer from Ali|500.0": "Salary",
#   "2026-05-15|Transfer from Ali|50.0":  "Refund"
# }

def make_inflow_key(row):
    return f"{row['Date']}|{row['Description']}|{row['Amount']}"

def apply_inflow_overrides(df):
    # Called once when a CSV is first loaded — restores previously saved categories
    for idx, row in df.iterrows():
        key = make_inflow_key(row)
        if key in inflow_overrides:
            df.at[idx, "Category"] = inflow_overrides[key]
    return df

def save_inflow_overrides():
    with open("inflow_overrides.json", "w") as f:
        json.dump(inflow_overrides, f)

---
## Section 7 — Normalising a CSV into Standard Columns

Every bank has different column names. The rest of the app only ever needs three columns: `Date`, `Description`, `Amount`. `normalise_df` is the bridge — it takes any bank's raw DataFrame and returns one with exactly those three columns.

### 7a — `normalise_df`

The `fmt` dict tells it which raw columns to read and how.

In [ ]:
def normalise_df(df, fmt):
    df = df.copy()  # never modify the original

    # ── Date ──────────────────────────────────────────────────────────────────
    # Try the stored format string first.
    # If the bank changed its export format (e.g. Revolut switched date style),
    # fall back to 'mixed' which lets pandas infer the format per row.
    try:
        df["Date"] = pd.to_datetime(df[fmt["date_col"]], format=fmt["date_format"]).dt.date
    except ValueError:
        df["Date"] = pd.to_datetime(df[fmt["date_col"]], format="mixed", dayfirst=True).dt.date
    # .dt.date strips the time component, giving a plain date object (2026-05-01)

    # ── Description ───────────────────────────────────────────────────────────
    df["Description"] = df[fmt["description_col"]].astype(str).str.strip()

    # ── Amount ────────────────────────────────────────────────────────────────
    if "debit_col" in fmt and "credit_col" in fmt:
        # Banks like HSBC split payments and receipts into two columns.
        # Debit = money out (positive number), Credit = money in (positive number).
        # We convert to: negative = outflow, positive = inflow.
        debit  = pd.to_numeric(df[fmt["debit_col"]].astype(str).str.replace(",", ""), errors="coerce").fillna(0)
        credit = pd.to_numeric(df[fmt["credit_col"]].astype(str).str.replace(",", ""), errors="coerce").fillna(0)
        df["Amount"] = credit - debit  # credit positive, debit negative
    else:
        # Most banks use a single signed amount column.
        # .str.replace(",", "") handles amounts like "1,234.56"
        # errors="coerce" turns unparseable values into NaN instead of crashing
        df["Amount"] = pd.to_numeric(df[fmt["amount_col"]].astype(str).str.replace(",", ""), errors="coerce")

    # Keep only the three standard columns and drop rows with no amount
    return df[["Date", "Description", "Amount"]].dropna(subset=["Amount"])

### 7b — `load_transactions`

A thin wrapper that chains: read CSV → normalise → categorise → return.

In [ ]:
def load_transactions(file, fmt):
    try:
        df = pd.read_csv(file)
        df.columns = [col.strip() for col in df.columns]  # remove accidental whitespace
        df = normalise_df(df, fmt)
        return categorize_transactions(df)
    except Exception as e:
        st.error(f"Error processing file: {str(e)}")
        return None  # caller must check for None

### 7c — `get_month_info`

Determines which month a CSV belongs to. We use the **most frequent month** across all dates rather than the first date, because a monthly statement often starts a day or two before the calendar month (e.g. April 30 in a May statement).

In [ ]:
def get_month_info(df):
    # Convert every date to a Period (e.g. 2026-05)
    # .value_counts() counts how many rows fall in each period
    # .idxmax() returns the period with the highest count
    dominant = pd.to_datetime(df["Date"]).dt.to_period("M").value_counts().idxmax()
    dt = dominant.to_timestamp()  # Period → Timestamp so we can use strftime
    return dt.strftime("%Y-%m"), dt.strftime("%B %Y")
    # Returns: ("2026-05", "May 2026")
    #           ↑ used as dict key    ↑ used as tab label

---
## Section 8 — Session State for Months

All processed month data is stored in `st.session_state.months`. This is the central data store for the entire app.

```python
st.session_state.months = {
    "2026-04": {
        "bank":        "Revolut",
        "month_label": "April 2026",
        "filename":    "revolut_apr.csv",
        "outflow_df":  <DataFrame>,   # Amount < 0 rows with Category column
        "inflow_df":   <DataFrame>,   # Amount > 0 rows with Category column
    },
    "2026-05": { ... }
}
```

**Why `"YYYY-MM"` as the key?**
- It sorts chronologically as a plain string (`"2026-04" < "2026-05"`)
- It is derived directly from the date data — no manual input needed
- It is unique per month (two CSVs for the same month would overwrite each other, which is the right behaviour)

**Why store `filename`?**
When the user removes a file from the uploader, we need to know which month entry to delete. The only information we have is the filename, so we store it alongside the data.

In [ ]:
if "months" not in st.session_state:
    st.session_state.months = {}

---
## Section 9 — Column Mapping UI (Unknown Banks)

When neither `detect_bank` nor `bank_mappings.json` recognises a CSV, we show a UI that lets the user manually map the columns. Once saved, the fingerprint is stored in `bank_mappings.json` so this step is never needed again for that bank.

The function takes a `suffix` parameter (the filename) to make all its widget keys unique — essential when multiple unknown files are uploaded at once.

In [ ]:
DATE_FORMAT_OPTIONS = {
    "dd/mm/YYYY HH:MM  (e.g. 01/06/2026 14:30)": "%d/%m/%Y %H:%M",
    "dd/mm/YYYY        (e.g. 01/06/2026)":        "%d/%m/%Y",
    "YYYY-MM-DD        (e.g. 2026-06-01)":        "%Y-%m-%d",
    "mm/dd/YYYY        (e.g. 06/01/2026)":        "%m/%d/%Y",
    "dd-mm-YYYY        (e.g. 01-06-2026)":        "%d-%m-%Y",
    "dd MMM YYYY       (e.g. 01 Jun 2026)":       "%d %b %Y",
}

def show_mapping_ui(raw_df, suffix=""):
    st.warning("Bank format not recognised. Map your columns below.")
    st.dataframe(raw_df.head(3), use_container_width=True, hide_index=True)

    cols = list(raw_df.columns)
    c1, c2, c3, c4 = st.columns(4)

    # Each selectbox gets a unique key by including the filename as suffix
    date_col   = c1.selectbox("Date column",        cols, key=f"map_date_{suffix}")
    desc_col   = c2.selectbox("Description column", cols, key=f"map_desc_{suffix}")
    amount_col = c3.selectbox("Amount column",      cols, key=f"map_amount_{suffix}")
    fmt_label  = c4.selectbox("Date format",        list(DATE_FORMAT_OPTIONS.keys()), key=f"map_fmt_{suffix}")

    if st.button("Save & Continue", type="primary", key=f"map_btn_{suffix}"):
        return {
            "date_col":        date_col,
            "date_format":     DATE_FORMAT_OPTIONS[fmt_label],
            "description_col": desc_col,
            "amount_col":      amount_col,
        }
    return None  # user hasn't clicked Save yet

---
## Section 10 — `render_month`: The Per-Month Tab Content

This function is called once for each uploaded month. It renders everything inside a month's tab. Because it can be called multiple times per rerun, every widget key is suffixed with `month_key`.

### 10a — Date range filter with guard

The guard prevents the picker from going more than 2 days outside the detected calendar month. This catches statements that start/end a day or two outside the month boundary.

`pd.offsets.MonthEnd(0)` is a pandas offset that snaps a date to the last day of its month.
```
pd.to_datetime("2026-05") + pd.offsets.MonthEnd(0)  →  2026-05-31
```

In [ ]:
def render_month(month_key):
    month = st.session_state.months[month_key]

    # Derive the hard boundaries for this calendar month
    month_start = pd.to_datetime(month_key).date()                              # e.g. 2026-05-01
    month_end   = (pd.to_datetime(month_key) + pd.offsets.MonthEnd(0)).date()  # e.g. 2026-05-31
    guard_min   = month_start - pd.Timedelta(days=2)                            # e.g. 2026-04-29
    guard_max   = month_end   + pd.Timedelta(days=2)                            # e.g. 2026-06-02

    # Find the actual min/max dates from the data, clamped to the guard
    all_dates = [pd.to_datetime(d) for d in
                 list(month["outflow_df"]["Date"]) + list(month["inflow_df"]["Date"])]
    min_date  = max(min(all_dates).date(), guard_min)  # can't go before guard_min
    max_date  = min(max(all_dates).date(), guard_max)  # can't go after  guard_max

    date_range = st.date_input(
        "Select date range",
        value=(min_date, max_date),  # default = full data range
        min_value=guard_min,         # user cannot pick before this
        max_value=guard_max,         # user cannot pick after this
        key=f"dr_{month_key}",
    )

    # Compute filtered views — these are passed to the data editors
    if len(date_range) == 2:          # user has picked both start and end
        s, e = date_range
        out = month["outflow_df"][(month["outflow_df"]["Date"] >= s) & (month["outflow_df"]["Date"] <= e)]
        inf = month["inflow_df"] [(month["inflow_df"]["Date"]  >= s) & (month["inflow_df"]["Date"]  <= e)]
    else:                             # user cleared the first date — avoid crash
        out = month["outflow_df"]
        inf = month["inflow_df"]

### 10b — Data editor and Apply Changes

`st.data_editor` renders an editable table. The user can change the Category dropdown per row. The function returns a DataFrame (`edited_outflow`) with whatever the user has typed/selected — but **nothing is saved until Apply Changes is clicked**.

**Why boolean indexing preserves indices:**
When you filter a DataFrame with `df[df["Amount"] < 0]`, the resulting DataFrame keeps the **original row indices** from the parent. So when we pass the filtered `out` to the data editor and iterate `edited_outflow.iterrows()`, the `idx` values correspond to the correct rows in the full `month["outflow_df"]`. This lets us update the right row even when the editor is only showing a date-filtered subset.

In [ ]:
    sub1, sub2 = st.tabs(["Cash Outflow (Payments)", "Cash Inflow (Transfers In)"])

    with sub1:
        st.subheader("Categorise Outflows")
        st.caption("Click a row in the Category column, choose from the dropdown, then hit Apply Changes.")

        edited_outflow = st.data_editor(
            out[["Date", "Description", "Amount", "Category"]],
            column_config={
                "Date":     st.column_config.DateColumn("Date", format="DD/MM/YYYY"),
                "Amount":   st.column_config.NumberColumn("Amount", format="%.2f"),
                "Category": st.column_config.SelectboxColumn(
                    "Category",
                    options=list(st.session_state.categories.keys())
                ),
            },
            hide_index=True,
            use_container_width=True,
            key=f"outflow_editor_{month_key}",
        )

        if st.button("Apply Changes", type="primary", key=f"apply_out_{month_key}"):
            for idx, row in edited_outflow.iterrows():
                # Only process rows the user actually changed
                if row["Category"] == month["outflow_df"].at[idx, "Category"]:
                    continue
                # Update the stored DataFrame
                month["outflow_df"].at[idx, "Category"] = row["Category"]
                # Save the description as a keyword so future uploads auto-categorise it
                add_keyword_to_category(row["Category"], row["Description"])

    with sub2:
        st.subheader("Categorise Inflows")
        edited_inflow = st.data_editor(
            inf[["Date", "Description", "Amount", "Category"]],
            column_config={
                "Date":     st.column_config.DateColumn("Date", format="DD/MM/YYYY"),
                "Amount":   st.column_config.NumberColumn("Amount", format="%.2f"),
                "Category": st.column_config.SelectboxColumn(
                    "Category",
                    options=list(st.session_state.categories.keys())
                ),
            },
            hide_index=True,
            use_container_width=True,
            key=f"inflow_editor_{month_key}",
        )

        if st.button("Apply Changes", type="primary", key=f"apply_inf_{month_key}"):
            for idx, row in edited_inflow.iterrows():
                month["inflow_df"].at[idx, "Category"] = row["Category"]
                inflow_overrides[make_inflow_key(row)] = row["Category"]  # save per-transaction
            save_inflow_overrides()

### 10c — Expense summary and charts

**Placed after both sub-tabs** so it is always visible regardless of which tab is active.

**Why re-derive `out` and `inf` here?**
The script runs top to bottom. `out` and `inf` were computed before the Apply Changes button handlers ran. If the user just clicked Apply, the handlers updated `month["outflow_df"]` — but `out` still holds the pre-click snapshot. Re-filtering here reads the now-updated DataFrame, so the summary reflects the change in the same rerun.

**The pandas operations:**
- `.groupby("Category")["Amount"].sum()` — sum the Amount column for each unique Category value
- `.abs()` — convert negative outflow amounts to positive for display
- `.count()` — count how many transactions fall in each category
- `.merge(..., how="outer")` — join two DataFrames, keeping all rows from both even if there's no match (outer join). Unmatched values become NaN, filled with `.fillna(0)`

In [ ]:
    # Re-derive after button handlers have run
    if len(date_range) == 2:
        s, e = date_range
        out = month["outflow_df"][(month["outflow_df"]["Date"] >= s) & (month["outflow_df"]["Date"] <= e)]
        inf = month["inflow_df"] [(month["inflow_df"]["Date"]  >= s) & (month["inflow_df"]["Date"]  <= e)]
    else:
        out = month["outflow_df"]
        inf = month["inflow_df"]

    st.subheader("Expense Summary")

    outflow_totals = out.groupby("Category")["Amount"].sum().abs().reset_index()
    outflow_totals.columns = ["Category", "Outflows"]
    # reset_index() converts the groupby result back to a regular DataFrame
    # with Category as a normal column instead of the index

    outflow_counts = out.groupby("Category")["Amount"].count().reset_index()
    outflow_counts.columns = ["Category", "Quantity"]

    inflow_totals = inf.groupby("Category")["Amount"].sum().reset_index()
    inflow_totals.columns = ["Category", "Inflows"]

    # Merge all three into one summary table
    category_totals = outflow_totals.merge(outflow_counts, on="Category", how="left")
    category_totals = category_totals.merge(inflow_totals, on="Category", how="outer").fillna(0)
    category_totals["Expenses"] = category_totals["Outflows"] - category_totals["Inflows"]
    category_totals["Quantity"] = category_totals["Quantity"].astype(int)
    category_totals = category_totals[["Category", "Quantity", "Outflows", "Inflows", "Expenses"]]
    category_totals = category_totals.sort_values("Expenses", ascending=False)

### 10d — Charts

Three charts are shown:

1. **Donut chart** — proportion of total spend per category. `hole=0.35` makes it a donut instead of a solid pie. `textinfo="percent+label"` puts both the label and percentage inside each slice.

2. **Horizontal bar** — average spend per transaction per category. Sorted ascending so the largest category is at the top. Using `orientation="h"` with categories on the y-axis means labels never overlap regardless of length.

3. **Grouped bar** — total Outflows, Inflows, and Net side-by-side per category. Uses `pd.melt()` to convert from wide format (one column per type) to long format (one row per type per category), which is what Plotly needs for a grouped bar.

**`pd.melt` explained:**
```
Wide format (what we have):
  Category  Outflows  Inflows  Net Total
  Food      94.77     0        94.77

Long format (what px.bar needs for color grouping):
  Category  Type       Amount
  Food      Outflows   94.77
  Food      Inflows    0.00
  Food      Net Total  94.77
```

In [ ]:
    chart_col1, chart_col2 = st.columns(2)  # side-by-side layout

    with chart_col1:
        fig = px.pie(
            category_totals[category_totals["Expenses"] > 0],  # exclude zero-expense rows
            values="Expenses",
            names="Category",
            title="Expenses by Category",
            hole=0.35,              # donut style
            template="plotly_dark",
        )
        fig.update_traces(textposition="inside", textinfo="percent+label")
        fig.update_layout(showlegend=False, paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)")
        st.plotly_chart(fig, use_container_width=True)

    with chart_col2:
        avg_data = category_totals[category_totals["Quantity"] > 0].copy()
        avg_data["Average"] = avg_data["Outflows"] / avg_data["Quantity"]
        avg_data = avg_data.sort_values("Average", ascending=True)  # largest at top
        fig_bar = px.bar(
            avg_data,
            x="Average", y="Category",
            orientation="h",
            title="Average Expense per Category",
            text=avg_data["Average"].apply(lambda x: f"£{x:.2f}"),
            template="plotly_dark",
        )
        fig_bar.update_traces(textposition="outside")
        fig_bar.update_layout(xaxis_title="Amount (£)", yaxis_title="",
                               paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)")
        st.plotly_chart(fig_bar, use_container_width=True)

    # Grouped bar: Outflows / Inflows / Net per category
    total_expense = out["Amount"].abs().sum()
    total_inflows = inf["Amount"].sum()
    net_total     = total_expense - total_inflows

    per_category = category_totals.rename(columns={"Expenses": "Net Total"})
    summary_row  = pd.DataFrame([{"Category": "TOTAL", "Outflows": total_expense,
                                   "Inflows": total_inflows, "Net Total": net_total}])

    bar_source = pd.concat([per_category[["Category", "Outflows", "Inflows", "Net Total"]], summary_row],
                            ignore_index=True)

    # melt: wide → long format for grouped bar
    bar_melted = bar_source.melt(
        id_vars="Category",
        value_vars=["Outflows", "Inflows", "Net Total"],
        var_name="Type",
        value_name="Amount",
    )

    fig2 = px.bar(bar_melted, x="Category", y="Amount", color="Type", barmode="group",
                  title="Outflows, Inflows & Net Total by Category",
                  text=bar_melted["Amount"].apply(lambda x: f"£{x:.2f}"),
                  template="plotly_dark")
    fig2.update_traces(textposition="outside")
    fig2.update_layout(yaxis_title="Amount (£)", paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)")
    st.plotly_chart(fig2, use_container_width=True)

---
## Section 11 — `render_comparison`: Multi-Month Analysis

This function is called for the Compare Months tab. It builds a combined view across all uploaded months and shows four charts.

### Building the monthly summary DataFrame
We loop over all stored months and compute total outflows/inflows/net for each, producing a row per month.

In [ ]:
def render_comparison():
    if len(st.session_state.months) < 2:
        st.info("Upload at least 2 months of data to see a comparison.")
        return

    sorted_months = sorted(st.session_state.months.items())  # chronological order

    # ── Monthly totals ─────────────────────────────────────────────────────────
    summary_rows = []
    for _, month in sorted_months:
        out_total = month["outflow_df"]["Amount"].abs().sum()
        inf_total = month["inflow_df"]["Amount"].sum()
        summary_rows.append({
            "Month":    month["month_label"],
            "Outflows": round(out_total, 2),
            "Inflows":  round(inf_total, 2),
            "Net":      round(out_total - inf_total, 2),
        })
    monthly = pd.DataFrame(summary_rows)

    # ── Key metric cards ───────────────────────────────────────────────────────
    # st.metric shows a headline number with an optional delta below it
    total_spend = monthly["Outflows"].sum()
    avg_spend   = monthly["Outflows"].mean()
    max_row     = monthly.loc[monthly["Outflows"].idxmax()]
    min_row     = monthly.loc[monthly["Outflows"].idxmin()]

    m1, m2, m3, m4 = st.columns(4)
    m1.metric("Total Spend (All Months)", f"£{total_spend:,.2f}")
    m2.metric("Avg Monthly Spend",        f"£{avg_spend:,.2f}")
    m3.metric("Highest Spend Month",      max_row["Month"],
              delta=f"£{max_row['Outflows']:,.2f}", delta_color="inverse")  # red = high spend
    m4.metric("Lowest Spend Month",       min_row["Month"],
              delta=f"£{min_row['Outflows']:,.2f}", delta_color="inverse")

    st.divider()

    # ── Chart 1: Monthly overview grouped bar ──────────────────────────────────
    melted = monthly.melt(id_vars="Month", value_vars=["Outflows", "Inflows", "Net"],
                          var_name="Type", value_name="Amount")
    fig1 = px.bar(melted, x="Month", y="Amount", color="Type", barmode="group",
                  title="Monthly Overview — Outflows, Inflows & Net",
                  text=melted["Amount"].apply(lambda x: f"£{x:,.2f}"),
                  template="plotly_dark")
    fig1.update_traces(textposition="outside")
    fig1.update_layout(yaxis_title="Amount (£)", paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)")
    st.plotly_chart(fig1, use_container_width=True)

    # ── Chart 2: Category spend per month stacked bar ─────────────────────────
    cat_rows = []
    for _, month in sorted_months:
        for cat, amt in month["outflow_df"].groupby("Category")["Amount"].sum().abs().items():
            cat_rows.append({"Month": month["month_label"], "Category": cat, "Amount": round(amt, 2)})
    cat_df = pd.DataFrame(cat_rows)

    fig2 = px.bar(cat_df, x="Month", y="Amount", color="Category", barmode="stack",
                  title="Spending by Category per Month", template="plotly_dark")
    fig2.update_layout(yaxis_title="Amount (£)", paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)")
    st.plotly_chart(fig2, use_container_width=True)

    col1, col2 = st.columns(2)

    with col1:
        # ── Chart 3: Top category trends line chart ────────────────────────────
        top_cats = cat_df.groupby("Category")["Amount"].sum().nlargest(6).index.tolist()
        # nlargest(6) finds the 6 categories with highest total spend across all months
        trend_df = cat_df[cat_df["Category"].isin(top_cats)]
        fig3 = px.line(trend_df, x="Month", y="Amount", color="Category",
                       title="Top Category Trends", markers=True, template="plotly_dark")
        fig3.update_layout(yaxis_title="Amount (£)", paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)")
        st.plotly_chart(fig3, use_container_width=True)

    with col2:
        # ── Chart 4: Month-over-month % change ────────────────────────────────
        monthly["MoM %"] = monthly["Outflows"].pct_change() * 100
        # pct_change() computes (current - previous) / previous
        # The first row is always NaN (no previous month to compare to)
        mom_df = monthly.dropna(subset=["MoM %"])

        fig4 = px.bar(mom_df, x="Month", y="MoM %",
                      title="Month-over-Month Spend Change (%)",
                      text=mom_df["MoM %"].apply(lambda x: f"{x:+.1f}%"),
                      color="MoM %",
                      color_continuous_scale=["#2ecc71", "#e2e2e2", "#e74c3c"],
                      color_continuous_midpoint=0,  # 0% = white, negative = green, positive = red
                      template="plotly_dark")
        fig4.update_traces(textposition="outside")
        fig4.update_layout(yaxis_title="Change (%)", coloraxis_showscale=False,
                            paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)")
        st.plotly_chart(fig4, use_container_width=True)

---
## Section 12 — `main()`: Orchestrating Everything

This is the entry point. It runs on every rerun and does five things in order:

1. Show the file uploader
2. Clean up months for files that were removed
3. Process any newly uploaded files
4. Show the mapping UI for unrecognised files
5. Build the tab layout and call `render_month` / `render_comparison`

In [ ]:
def main():
    st.title("Simple Finance Dashboard")

    # accept_multiple_files=True returns a list of file objects
    uploaded_files = st.file_uploader(
        "Upload your transaction CSV files (one per month)",
        type=["csv"],
        accept_multiple_files=True,
    )

    if not uploaded_files:
        st.session_state.months = {}  # clear everything if no files
        return

In [ ]:
    # ── Step 2: Remove months for deselected files ─────────────────────────────
    current_names = {f.name for f in uploaded_files}

    for mk in list(st.session_state.months.keys()):
        # list() is required — you can't delete from a dict while iterating it
        if st.session_state.months[mk]["filename"] not in current_names:
            del st.session_state.months[mk]

In [ ]:
    # ── Step 3: Process new files ──────────────────────────────────────────────
    bank_mappings   = load_bank_mappings()
    processed_names = {v["filename"] for v in st.session_state.months.values()}

    for uploaded_file in uploaded_files:
        if uploaded_file.name in processed_names:
            continue  # already processed on a previous rerun — skip

        raw_df = pd.read_csv(uploaded_file)
        raw_df.columns = [col.strip() for col in raw_df.columns]
        uploaded_file.seek(0)  # reset file pointer so load_transactions can read it again

        fingerprint = make_fingerprint(raw_df.columns)
        bank_name   = detect_bank(raw_df.columns)
        fmt         = None

        if bank_name:
            fmt = BANK_FORMATS[bank_name]
        elif fingerprint in bank_mappings:
            fmt       = bank_mappings[fingerprint]
            bank_name = "Custom"

        if fmt is None:
            continue  # unknown bank — handled by mapping UI in step 4

        df = load_transactions(uploaded_file, fmt)
        if df is not None:
            month_key, month_label = get_month_info(df)
            st.session_state.months[month_key] = {
                "bank":        bank_name,
                "month_label": month_label,
                "filename":    uploaded_file.name,
                "outflow_df":  df[df["Amount"] < 0].copy(),
                "inflow_df":   apply_inflow_overrides(df[df["Amount"] > 0].copy()),
            }

In [ ]:
    # ── Step 4: Mapping UI for unrecognised files ──────────────────────────────
    processed_names = {v["filename"] for v in st.session_state.months.values()}

    for uploaded_file in uploaded_files:
        if uploaded_file.name in processed_names:
            continue
        uploaded_file.seek(0)  # file pointer may be at end — reset before reading
        raw_df = pd.read_csv(uploaded_file)
        raw_df.columns = [col.strip() for col in raw_df.columns]
        fingerprint = make_fingerprint(raw_df.columns)

        if detect_bank(raw_df.columns) is None and fingerprint not in bank_mappings:
            st.subheader(f"Map columns for `{uploaded_file.name}`")
            fmt = show_mapping_ui(raw_df, suffix=uploaded_file.name)
            if fmt is not None:
                bank_mappings[fingerprint] = fmt 
                save_bank_mappings(bank_mappings)
                st.rerun()  # force a full rerun so the file is processed in step 3

In [ ]:
    # ── Step 5: Build tabs ─────────────────────────────────────────────────────
    if not st.session_state.months:
        return

    sorted_months = sorted(st.session_state.months.items())
    # sorted() on "YYYY-MM" strings gives chronological order automatically

    for _, month in sorted_months:
        st.success(f"Detected: **{month['bank']}** — {month['month_label']} loaded")

    # Append Compare Months as the last tab
    all_tabs = st.tabs([m["month_label"] for _, m in sorted_months] + ["Compare Months"])

    # Render month tabs (all_tabs[:-1] excludes the last Compare tab)
    for tab, (month_key, _) in zip(all_tabs[:-1], sorted_months):
        with tab:
            render_month(month_key)

    # Render comparison tab (last element)
    with all_tabs[-1]:
        render_comparison()

main()

---
## Quick Reference — Patterns Used Throughout

| Pattern | Code | Why |
|---|---|---|
| Guard session state init | `if "x" not in st.session_state:` | Prevents reset on every rerun |
| Unique widget keys | `key=f"btn_{month_key}"` | Required when a function is called multiple times |
| Reset file pointer | `uploaded_file.seek(0)` | After one `pd.read_csv`, the pointer is at the end — reset before reading again |
| Safe dict delete | `for k in list(d.keys()): del d[k]` | Can't delete from dict while iterating it directly |
| Preserved index filter | `df[df["Amount"] < 0]` | Keeps original row indices — lets you update the parent df by index |
| Wide → long format | `df.melt(id_vars=..., value_vars=...)` | Required by Plotly for grouped/stacked bars |
| Re-derive after buttons | recompute `out`/`inf` after Apply Changes block | Buttons update session state; variables computed before them are stale |
| Most frequent month | `.dt.to_period("M").value_counts().idxmax()` | Avoids mislabelling a May statement that starts on April 30 |
| Month boundary | `pd.to_datetime(key) + pd.offsets.MonthEnd(0)` | Snaps to the last day of the month |

---
## Section 14 — Clearing a Text Input After Submit

Streamlit widgets are **controlled** — their value is tied to their `key` in `st.session_state`. There is no `clear()` method. The only reliable way to reset a text input is to give it a **different key**, which causes Streamlit to render a completely new widget with an empty default value.

### The version-counter pattern

Store an integer counter in session state. Append it to the widget's key. Each time you want to clear the widget, increment the counter — the key changes, the widget is recreated empty.

```
sidebar_new_cat_0   →  sidebar_new_cat_1   →  sidebar_new_cat_2
(first render)          (after first add)       (after second add)
```

This is the standard approach because:
- Streamlit offers no direct widget mutation API
- Setting `st.session_state["sidebar_new_cat_0"] = ""` would work on the next rerun, but the key must still be valid for the current widget — and once incremented the old key no longer exists

### Why only increment on success?

If the user types a duplicate name the warning stays visible and the text stays in place so they can correct it. Incrementing only on a successful add keeps the UX intuitive.

In [ ]:
# ui/sidebar.py — version-counter pattern for clearing text input

if "sidebar_new_cat_v" not in st.session_state:
    st.session_state.sidebar_new_cat_v = 0

# Key changes every time the counter increments → widget is recreated empty
new_category = st.text_input(
    "Category name",
    placeholder="e.g. Groceries",
    key=f"sidebar_new_cat_{st.session_state.sidebar_new_cat_v}",
)

if st.button("Add Category", type="primary", key="sidebar_add_cat") and new_category:
    if new_category not in st.session_state.categories:
        st.session_state.categories[new_category] = []
        save_categories()
        st.session_state.sidebar_new_cat_v += 1  # ← increments key → clears input on next rerun
        st.success(f"**{new_category}** added.")
    else:
        st.warning(f"**{new_category}** already exists.")  # no increment — text stays for correction

---
## Section 15 — 50/30/20 Rule: Three-Bucket System with Savings

The original 50/30/20 tab had two buckets (Needs and Wants) and calculated Savings as a passive leftover. This was extended to add an explicit **Savings** bucket so the user can assign categories like Pension or ISA to it directly.

### Three-way mutual exclusion

Each multiselect's available options must exclude whatever is already selected in the other two. With three widgets rendered in a single rerun, we face a sequencing problem:

- **Needs** is rendered first — it can only use previous-rerun session state to know what's in Wants/Savings
- **Wants** is rendered second — it can use the *current* return value of Needs (already computed this rerun), but only previous-rerun state for Savings
- **Savings** is rendered last — it can use the current return values of both Needs and Wants

This asymmetry is intentional. There is a one-rerun lag for Savings → Wants exclusion, but in practice it is invisible to the user.

```
Needs options   = all_cats - prev_wants  - prev_savings
Wants options   = all_cats - needs_cats  - prev_savings   (needs_cats = this rerun's return)
Savings options = all_cats - needs_cats  - wants_cats     (wants_cats = this rerun's return)
```

### Savings calculation

The savings bucket collects two things:
1. **Explicit savings** — the sum of outflows in categories assigned to the Savings bucket (e.g. pension contributions, ISA top-ups)
2. **Leftover** — whatever remains from take-home pay after Needs, Wants, and explicit Savings spending

```
leftover       = max(salary - needs_spend - wants_spend - explicit_savings, 0)
total_savings  = explicit_savings + leftover
```

This means if your Needs + Wants + explicit Savings already exceed your salary, leftover is 0 — no negative savings.

### Unassigned categories: info not warning

Changed from `st.warning` (yellow) to `st.info` (blue) because having unassigned categories is not an error state — the user may still be working through them. The message lists the unassigned categories by name rather than demanding action.

In [ ]:
# ui/budget_rule.py — three-bucket multiselects with mutual exclusion

available_cats = [c for c in st.session_state.categories if c != "Uncategorized"]

prev_wants   = st.session_state.get("5030_wants",   [])
prev_savings = st.session_state.get("5030_savings", [])

# Each column excludes what's already committed in the other two
needs_options = [c for c in available_cats if c not in prev_wants and c not in prev_savings]

col1, col2, col3 = st.columns(3)
with col1:
    st.markdown("**Needs** — essentials")
    needs_cats = st.multiselect("needs_select", needs_options, key="5030_needs",
                                label_visibility="collapsed")
with col2:
    wants_options = [c for c in available_cats if c not in needs_cats and c not in prev_savings]
    st.markdown("**Wants** — lifestyle")
    wants_cats = st.multiselect("wants_select", wants_options, key="5030_wants",
                                label_visibility="collapsed")
with col3:
    savings_options = [c for c in available_cats if c not in needs_cats and c not in wants_cats]
    st.markdown("**Savings** — pension, ISA, etc.")
    savings_cats = st.multiselect("savings_select", savings_options, key="5030_savings",
                                  label_visibility="collapsed")

unassigned = [c for c in available_cats
              if c not in needs_cats and c not in wants_cats and c not in savings_cats]
if unassigned:
    st.info(f"Not yet assigned: **{', '.join(unassigned)}**")

# ── Savings calculation ────────────────────────────────────────────────────────
# salary comes from: st.number_input(..., key="5030_salary")
# The key is required so sidebar.py can read it via st.session_state.get("5030_salary", 0)

avg_by_cat = all_out.groupby("Category")["Amount"].sum().abs() / n_months

actual_needs            = avg_by_cat[avg_by_cat.index.isin(needs_cats)].sum()
actual_wants            = avg_by_cat[avg_by_cat.index.isin(wants_cats)].sum()
actual_savings_explicit = avg_by_cat[avg_by_cat.index.isin(savings_cats)].sum()
actual_savings_leftover = max(salary - actual_needs - actual_wants - actual_savings_explicit, 0.0)
actual_savings          = actual_savings_explicit + actual_savings_leftover

---
## Section 16 — Multiple Bank Files per Month

The original app stored one file per month. If you uploaded a Monzo CSV and a Revolut CSV both covering June, the second one would overwrite the first. The fix allows multiple files to contribute to the same month's data.

### Data structure change

The month entry changed from storing a single filename and bank name to storing a **set of filenames** and a **list of banks**:

```python
# Before
st.session_state.months["2026-06"] = {
    "bank":     "Monzo",
    "filename": "monzo_june.csv",
    "outflow_df": ...,
    "inflow_df":  ...,
}

# After
st.session_state.months["2026-06"] = {
    "banks":     ["Monzo", "Revolut"],
    "filenames": {"monzo_june.csv", "revolut_june.csv"},  # set, not string
    "outflow_df": ...,   # concatenated from both files
    "inflow_df":  ...,   # concatenated from both files
}
```

A **set** is used for `filenames` because order doesn't matter and set membership tests (`in`) and subset checks (`.issubset()`) are efficient.

### Merging on upload

When a processed file's `month_key` already exists in session state, the new DataFrame is concatenated onto the existing one instead of replacing it:

```python
if month_key in st.session_state.months:
    existing = st.session_state.months[month_key]
    existing["outflow_df"] = pd.concat([existing["outflow_df"], new_out], ignore_index=True)
    existing["inflow_df"]  = pd.concat([existing["inflow_df"],  new_inf],  ignore_index=True)
    existing["filenames"].add(uploaded_file.name)
    existing["banks"].append(bank_name)
```

`ignore_index=True` resets the index to 0, 1, 2… after concatenation — important because both DataFrames start with index 0, so without this you'd have duplicate index values.

### Removal: invalidate and re-merge

You cannot "un-concatenate" a merged DataFrame. When any file from a month is removed, the whole month entry is deleted. The remaining files are still in the uploader, so they get re-processed and re-merged on the same rerun.

```python
for mk in list(st.session_state.months.keys()):
    # issubset: returns True only if ALL filenames are still uploaded
    if not st.session_state.months[mk]["filenames"].issubset(current_names):
        del st.session_state.months[mk]
```

### Processed names across all months

Previously `processed_names` was a simple set comprehension over one filename per month. With multiple filenames per month it becomes a nested set comprehension:

```python
# Before
processed_names = {v["filename"] for v in st.session_state.months.values()}

# After
processed_names = {name for v in st.session_state.months.values() for name in v["filenames"]}
```

---
## Section 17 — PDF Report Export

A **Generate Report** button in the sidebar produces a multi-page PDF containing:
- A cover page (banks, months, totals)
- Section 1: Full transaction log (all months, sorted by date)
- Section 2: Per-month expense summary table + pie chart + grouped bar chart
- Section 3: Month comparison table + overview bar + category trend line
- Section 4: 50/30/20 bucket table + bar chart + 40-year savings projection + milestone table

### Dependencies

Two new packages are required:

| Package | Purpose |
|---|---|
| `fpdf2` | Pure-Python PDF builder — adds pages, draws tables, embeds images |
| `kaleido` | Headless renderer that converts Plotly figures to PNG bytes |

Install: `pip install fpdf2 kaleido`

### Chart images in a PDF

Plotly charts are interactive HTML/JavaScript — they cannot be placed directly in a PDF. `kaleido` is a standalone binary (installed with `pip install kaleido`) that Plotly uses internally when you call `fig.to_image()`. It renders the figure as a static PNG.

```python
def _fig_png(fig, width=900, height=400):
    try:
        return fig.to_image(format="png", width=width, height=height, scale=1.5)
    except Exception:
        return None  # kaleido missing or render failed — skip the image gracefully
```

The PNG bytes are then embedded in the PDF via `fpdf2`'s `image()` method, which accepts a `BytesIO` object:

```python
pdf.image(io.BytesIO(img_bytes), w=PAGE_W)  # PAGE_W = 180mm (A4 minus margins)
```

### fpdf2 layout primitives

| Method | What it does |
|---|---|
| `pdf.add_page()` | Start a new A4 page |
| `pdf.set_font("Helvetica", "B", 13)` | Set font family, style (B=bold, I=italic), size in pt |
| `pdf.set_fill_color(r, g, b)` | Set the background colour for the next filled cell |
| `pdf.set_text_color(r, g, b)` | Set the text colour |
| `pdf.cell(w, h, text, border=1, fill=True)` | Draw a single cell of fixed width |
| `pdf.ln()` | Move to the next line |
| `pdf.multi_cell(0, h, text)` | Draw a cell that wraps to the full page width |
| `pdf.image(src, w=width)` | Embed an image (file path, bytes, or BytesIO) |
| `pdf.set_auto_page_break(auto=True, margin=15)` | Auto-insert a new page when content reaches 15mm from the bottom |

### Table rendering

Tables are drawn cell by cell. Each row is a loop over `(value, width)` pairs. Alternating rows use a light-grey fill (`242, 242, 242`) for readability — the `stripe` boolean toggles `fill=True/False`:

```python
def _table_row(pdf, values, widths, stripe):
    pdf.set_font("Helvetica", "", 7)
    pdf.set_fill_color(242, 242, 242)          # always set; only applied when fill=True
    for val, w in zip(values, widths):
        pdf.cell(w, 5, str(val), border="LRB", fill=stripe)
    pdf.ln()
```

`border="LRB"` draws Left, Right, and Bottom borders — no top border because the bottom of the previous row already forms the visual top boundary.

### Validation before download

The report is only meaningful when:
1. At least one month is uploaded
2. No transactions remain as *Uncategorized*
3. Needs and Wants categories are assigned in the 50/30/20 tab
4. Monthly take-home pay is greater than zero

The sidebar checks all four and shows `st.warning` for each failing condition. The Generate button is hidden until all conditions pass.

### Two-button UX and session state storage

Generating the PDF (which renders charts via kaleido) takes a few seconds. A naive approach would regenerate on every Streamlit rerun — wasteful and slow. Instead:

1. **Generate Report** button — triggers generation once and stores the bytes in `st.session_state["_report_pdf"]`
2. **Download Report PDF** button — appears after generation; reads from session state

```python
if st.button("Generate Report", key="gen_report_btn", type="primary"):
    with st.spinner("Building PDF — rendering charts..."):
        st.session_state["_report_pdf"] = generate_report_pdf(salary, needs_cats, wants_cats, savings_cats)

if "_report_pdf" in st.session_state:
    st.download_button("Download Report PDF",
                       data=st.session_state["_report_pdf"],
                       file_name=f"finance_report_{date.today().strftime('%Y-%m-%d')}.pdf",
                       mime="application/pdf")
```

If the user changes data (triggering new validation errors), `_report_pdf` is deleted from session state, forcing regeneration before a new download is offered.

### Reading the salary key from sidebar

The 50/30/20 salary `st.number_input` has `key="5030_salary"`. Without an explicit key, Streamlit auto-generates one that cannot be reliably read from other modules. With the explicit key, the sidebar can read it directly:

```python
salary = st.session_state.get("5030_salary", 0)
```

This cross-module session state access works because session state is global within a Streamlit app — any module that imports `streamlit as st` shares the same `st.session_state` object.